# Train eye classifier (ROI architecture)

Notebook này huấn luyện **image-classification model** cho mắt, không huấn luyện YOLO object detector.

Pipeline runtime tương ứng:

```text
Camera frame → face ROI → left/right eye ROI → classifier → Open / Closed
```

Dataset hiện tại chứa ảnh theo class (`Open`, `Closed`) nhưng không chứa tọa độ bounding box mắt hợp lệ. Vì vậy notebook không đọc thư mục `images/labels` và không gọi `model.plot()` trong runtime. Model đầu ra là xác suất class cho từng eye crop.


## 1. Cài đặt và tham số

In [ ]:
from pathlib import Path

CLASSES = ('Open', 'Closed')
SOURCE = None  # Có thể sửa cell discovery bên dưới bằng đường dẫn Kaggle thực tế.
PREPARED = Path('/kaggle/working/eye_classification')
RUNS = Path('/kaggle/working/runs_eye')
RUN_NAME = 'eye_classifier'


## 2. Tìm và kiểm tra dataset classification

Notebook tự tìm thư mục có dạng `train/Open`, `train/Closed`, `test/Open`, `test/Closed`. Nếu Kaggle mount tên khác, đặt `SOURCE` thành đường dẫn thực tế.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random
import shutil
import hashlib
from PIL import Image

SEED = 42
random.seed(SEED)

# Để None để tự tìm trong /kaggle/input. Có thể đặt đường dẫn thủ công nếu Kaggle
# dùng tên mount khác, ví dụ: Path('/kaggle/input/ten-dataset/dataset_new').
# Giữ None để tự tìm; nếu cần, sửa dòng này thành Path('...').
SOURCE = None
EXPECTED = ('train', 'test')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def has_classification_layout(root: Path, classes):
    return all(
        (root / split).is_dir()
        and all((root / split / cls).is_dir() for cls in classes)
        for split in EXPECTED
    )


def discover_source(classes):
    if SOURCE is not None:
        root = Path(SOURCE)
        if not has_classification_layout(root, classes):
            raise FileNotFoundError(
                f'SOURCE không có cấu trúc train/test/{classes}: {root}'
            )
        return root

    search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    candidates = []
    for base in search_roots:
        if not base.exists():
            continue
        for train_dir in base.rglob('train'):
            root = train_dir.parent
            if has_classification_layout(root, classes):
                candidates.append(root)
    # Loại trùng và ưu tiên đường dẫn nông hơn.
    candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
    if not candidates:
        raise FileNotFoundError(
            f'Không tìm thấy dataset classification có train/test và classes={classes}. '
            'Hãy đặt SOURCE thủ công ở cell này.'
        )
    print('Các dataset tìm thấy:')
    for p in candidates:
        print(' -', p)
    return candidates[0]


def image_files(folder):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def inventory(root, classes):
    counts = {}
    for split in EXPECTED:
        counts[split] = {cls: len(image_files(root / split / cls)) for cls in classes}
    return counts


In [ ]:
EYE_SOURCE = discover_source(CLASSES)
print('SOURCE =', EYE_SOURCE)
print(json.dumps(inventory(EYE_SOURCE, CLASSES), indent=2))


## 3. Tạo train/validation/test

Nguồn chỉ có `train` và `test`, nên cell này tách 20% của train thành validation theo từng class. Test giữ nguyên và không dùng để chọn threshold.

In [ ]:
def prepare_classification_dataset(source, destination, classes, val_fraction=0.20):
    """Create train/val/test folders from train/test class folders.

    The source has no validation split, so val is a stratified split of source/train.
    Test is copied without being used for model-selection thresholds.
    """
    if destination.exists():
        shutil.rmtree(destination)
    for split in ('train', 'val', 'test'):
        for cls in classes:
            (destination / split / cls).mkdir(parents=True, exist_ok=True)

    manifest = []
    for cls in classes:
        train_items = image_files(source / 'train' / cls)
        test_items = image_files(source / 'test' / cls)
        random.Random(SEED).shuffle(train_items)
        n_val = max(1, round(len(train_items) * val_fraction))
        groups = {
            'val': train_items[:n_val],
            'train': train_items[n_val:],
            'test': test_items,
        }
        for split, items in groups.items():
            for index, src in enumerate(items):
                # Prefix avoids collisions when different class folders reuse filenames.
                dst_name = f'{cls}_{index:06d}_{src.name}'
                dst = destination / split / cls / dst_name
                shutil.copy2(src, dst)
                manifest.append({'split': split, 'class': cls, 'source': str(src), 'copied': str(dst)})

    (destination / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return manifest


def verify_dataset(root, classes):
    report = {}
    for split in ('train', 'val', 'test'):
        report[split] = {}
        for cls in classes:
            files = image_files(root / split / cls)
            good = 0
            for path in files:
                try:
                    with Image.open(path) as im:
                        im.verify()
                    good += 1
                except Exception as exc:
                    raise RuntimeError(f'Ảnh lỗi: {path}: {exc}')
            report[split][cls] = good
    return report


In [ ]:
manifest = prepare_classification_dataset(EYE_SOURCE, PREPARED, CLASSES, val_fraction=0.20)
print(json.dumps(verify_dataset(PREPARED, CLASSES), indent=2))
print('Samples copied:', len(manifest))


## 4. Train eye classifier

Model nhận một eye crop 224×224 và trả về `Open` hoặc `Closed`. Đây là classification model, không có bounding box.

In [ ]:
# Cài dependency trong Kaggle nếu cần.
%pip install -q ultralytics

from ultralytics import YOLO
import torch

EPOCHS = 40
IMG_SIZE = 224
BATCH = 32
WORKERS = 2
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO('yolov8n-cls.pt')
model.train(
    data=str(PREPARED),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    seed=SEED,
    project=str(RUNS),
    name=RUN_NAME,
    patience=10,
    device=DEVICE,
    workers=WORKERS,
    pretrained=True,
)

BEST = RUNS / RUN_NAME / 'weights' / 'best.pt'
print('Best checkpoint:', BEST)


## 5. Đánh giá và lưu checkpoint

In [ ]:
# Đánh giá trên test một lần sau khi chốt hyperparameters.
best_model = YOLO(str(BEST))
test_metrics = best_model.val(
    data=str(PREPARED),
    split='test',
    imgsz=IMG_SIZE,
    project=str(RUNS),
    name=f'{RUN_NAME}_test',
    device=DEVICE,
)
print(test_metrics)

# In class mapping để runtime không bị đảo nhãn.
print('Class names:', best_model.names)
print('Checkpoint dùng cho camera:', BEST)


## 6. Hợp đồng input cho camera runtime

Runtime phải tự tạo hai ROI từ khuôn mặt, sau đó gọi classifier riêng cho mắt trái và mắt phải. Không đưa toàn frame vào model này và không dùng `.plot()` để vẽ box.

```text
frame → face/landmark ROI → left eye crop → eye classifier
                              → right eye crop → eye classifier
```

Class mapping phải lấy từ `best_model.names`, không tự giả định index nếu checkpoint trả về mapping khác.